# 🌦️ Analyse Exploratoire des Données — WeatherForYnov

**Objectif** : Explorer les données climatiques mensuelles de Météo France (1950–2026) pour identifier les tendances liées au changement climatique en France.

**Données** : Fichiers MENSQ (données mensuelles de synthèse climatologique) provenant de Météo France.

---

## 1. Imports et configuration

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

print("✅ Imports OK")

✅ Imports OK


## 2. Chargement et fusion des données

In [24]:
import glob
import os

# Charger tous les fichiers CSV de tous les départements
all_files = sorted(glob.glob("Data/MENSQ_*_*.csv"))
print(f"Fichiers trouvés : {len(all_files)}")

dfs = []
for f in all_files:
    # Extraire le numéro de département depuis le nom du fichier
    basename = os.path.basename(f)
    dept = basename.split('_')[1]
    tmp = pd.read_csv(f, sep=";", low_memory=False)
    tmp['DEPARTEMENT'] = dept
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)
print(f"Total : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Départements : {df['DEPARTEMENT'].nunique()} ({', '.join(sorted(df['DEPARTEMENT'].unique()))})")

Fichiers trouvés : 189
Total : 3,262,696 lignes × 163 colonnes
Départements : 95 (01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95)


In [25]:
df.head()

,NUM_POSTE,NOM_USUEL,LAT,LON,ALTI,AAAAMM,RR,QRR,NBRR,RR_ME,...,NEIGETOTAB,QNEIGETOTAB,NEIGETOTABDAT,NBJNEIGETOT1,NBJNEIGETOT10,NBJNEIGETOT30,NBJGREL,NBJORAG,NBJBROU,DEPARTEMENT
0,1014002,ARBENT,46.278167,5.669,534,202501,187.3,1.0,31.0,NaN,...,5.0,9.0,2.0,2.0,0.0,0.0,NaN,NaN,NaN,01
1,1014002,ARBENT,46.278167,5.669,534,202502,54.0,1.0,28.0,NaN,...,0.0,9.0,NaN,0.0,0.0,0.0,NaN,NaN,NaN,01
2,1014002,ARBENT,46.278167,5.669,534,202503,64.2,1.0,31.0,NaN,...,4.0,9.0,15.0,1.0,0.0,0.0,NaN,NaN,NaN,01
3,1014002,ARBENT,46.278167,5.669,534,202504,126.0,1.0,30.0,NaN,...,0.0,9.0,NaN,0.0,0.0,0.0,NaN,NaN,NaN,01
4,1014002,ARBENT,46.278167,5.669,534,202505,133.7,1.0,31.0,NaN,...,0.0,9.0,NaN,0.0,0.0,0.0,NaN,NaN,NaN,01


## 3. Exploration initiale et compréhension des variables

### Dictionnaire des colonnes clés

| Colonne | Description |
|---------|------------|
| `NUM_POSTE` | Identifiant de la station météo |
| `NOM_USUEL` | Nom de la station |
| `LAT` / `LON` | Coordonnées géographiques |
| `ALTI` | Altitude (m) |
| `AAAAMM` | Année et mois (format YYYYMM) |
| `RR` | Cumul mensuel de précipitations (mm) |
| `TX` | Température maximale moyenne (°C) |
| `TN` | Température minimale moyenne (°C) |
| `TM` | Température moyenne (°C) |
| `INST` | Durée d'insolation (heures) |
| `FFM` | Vitesse moyenne du vent (m/s) |
| `NBJGELEE` | Nombre de jours de gelée |
| `NBJTX30` | Nombre de jours avec T max ≥ 30°C |
| `NBJTX35` | Nombre de jours avec T max ≥ 35°C |
| `GLOT` | Rayonnement global (J/cm²) |

In [26]:
print("=== Info général ===")
print(f"Nombre de lignes   : {len(df):,}")
print(f"Nombre de colonnes : {df.shape[1]}")
print(f"Stations uniques   : {df['NUM_POSTE'].nunique()}")
print(f"Période            : {str(df['AAAAMM'].min())} → {str(df['AAAAMM'].max())}")
print(f"\n=== Types de données ===")
print(df.dtypes.value_counts())

=== Info général ===
Nombre de lignes   : 3,262,696
Nombre de colonnes : 163
Stations uniques   : 9405
Période            : 195001 → 202606

=== Types de données ===
float64    158
int64        3
object       2
Name: count, dtype: int64


## 4. Nettoyage et préparation des données

In [27]:
# Extraction année et mois depuis AAAAMM
df['AAAAMM'] = df['AAAAMM'].astype(str)
df['ANNEE'] = df['AAAAMM'].str[:4].astype(int)
df['MOIS'] = df['AAAAMM'].str[4:6].astype(int)
df['DATE'] = pd.to_datetime(df['AAAAMM'], format='%Y%m')

# Colonnes d'intérêt pour l'analyse climatique
cols_cles = ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'DATE', 'ANNEE', 'MOIS', 'DEPARTEMENT',
             'RR', 'TX', 'TN', 'TM', 'INST', 'FFM', 'NBJGELEE', 'NBJTX30', 'NBJTX35',
             'GLOT', 'NBJRR1', 'NBJRR10', 'HNEIGEFTOT', 'NBJNEIG']

cols_disponibles = [c for c in cols_cles if c in df.columns]
cols_manquantes = [c for c in cols_cles if c not in df.columns]
if cols_manquantes:
    print(f"⚠️ Colonnes manquantes : {cols_manquantes}")

print(f"✅ {len(cols_disponibles)} colonnes retenues pour l'analyse")
df_clean = df[cols_disponibles].copy()
df_clean.head()

✅ 23 colonnes retenues pour l'analyse



KeyboardInterrupt



In [ ]:
# Analyse des valeurs manquantes
cols_num = ['RR', 'TX', 'TN', 'TM', 'INST', 'FFM', 'NBJGELEE', 'NBJTX30', 'NBJTX35',
            'GLOT', 'NBJRR1', 'NBJRR10', 'HNEIGEFTOT', 'NBJNEIG']
cols_num = [c for c in cols_num if c in df_clean.columns]

missing = df_clean[cols_num].isnull().sum()
missing_pct = (missing / len(df_clean) * 100).round(1)

missing_df = pd.DataFrame({'Manquants': missing, '% Manquant': missing_pct})
missing_df = missing_df.sort_values('% Manquant', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
missing_df['% Manquant'].plot(kind='barh', ax=ax, color='coral')
ax.set_xlabel('% de valeurs manquantes')
ax.set_title('Taux de valeurs manquantes par variable climatique')
plt.tight_layout()
plt.show()

print(missing_df)

In [ ]:
# Statistiques descriptives des variables clés
df_clean[cols_num].describe().round(2)

## 5. Distributions des variables climatiques

In [ ]:
# Distribution des températures (TM, TX, TN)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, label, color in zip(axes, ['TN', 'TM', 'TX'],
                                  ['T° min (TN)', 'T° moyenne (TM)', 'T° max (TX)'],
                                  ['#3498db', '#2ecc71', '#e74c3c']):
    data = df_clean[col].dropna()
    ax.hist(data, bins=60, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(data.mean(), color='black', linestyle='--', label=f'Moyenne: {data.mean():.1f}°C')
    ax.set_title(label)
    ax.set_xlabel('°C')
    ax.legend()

plt.suptitle('Distribution des températures mensuelles', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des précipitations et insolation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_clean['RR'].dropna().plot(kind='hist', bins=60, ax=axes[0], color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('Distribution des précipitations mensuelles (RR)')
axes[0].set_xlabel('Précipitations (mm)')

df_clean['INST'].dropna().plot(kind='hist', bins=60, ax=axes[1], color='orange', alpha=0.7, edgecolor='white')
axes[1].set_title("Distribution de l'insolation mensuelle (INST)")
axes[1].set_xlabel('Heures')

plt.tight_layout()
plt.show()

## 6. Évolution temporelle — Tendances du changement climatique

In [ ]:
# Température moyenne annuelle — toutes stations France entière
tm_annuel = df_clean.groupby('ANNEE')['TM'].mean()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(tm_annuel.index, tm_annuel.values, color='gray', alpha=0.5, linewidth=0.8)

# Moyenne glissante sur 10 ans
rolling = tm_annuel.rolling(10, center=True).mean()
ax.plot(rolling.index, rolling.values, color='red', linewidth=2.5, label='Moyenne glissante 10 ans')

# Tendance linéaire
mask = tm_annuel.dropna()
z = np.polyfit(mask.index, mask.values, 1)
p = np.poly1d(z)
ax.plot(mask.index, p(mask.index), '--', color='darkred', linewidth=1.5,
        label=f'Tendance : {z[0]*10:+.2f}°C / décennie')

ax.set_title('Évolution de la température moyenne annuelle en France (tous départements)', fontsize=14)
ax.set_xlabel('Année')
ax.set_ylabel('Température moyenne (°C)')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Évolution des précipitations annuelles moyennes
rr_annuel = df_clean.groupby('ANNEE')['RR'].mean()

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(rr_annuel.index, rr_annuel.values, color='steelblue', alpha=0.6, width=0.8)
rolling_rr = rr_annuel.rolling(10, center=True).mean()
ax.plot(rolling_rr.index, rolling_rr.values, color='darkblue', linewidth=2.5, label='Moyenne glissante 10 ans')

ax.set_title('Évolution des précipitations mensuelles moyennes par année', fontsize=14)
ax.set_xlabel('Année')
ax.set_ylabel('Précipitations moyennes (mm/mois)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Évolution du nombre de jours de forte chaleur (≥30°C et ≥35°C) et de gelée
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title, color in zip(axes,
    ['NBJTX30', 'NBJTX35', 'NBJGELEE'],
    ['Jours T° max ≥ 30°C', 'Jours T° max ≥ 35°C', 'Jours de gelée'],
    ['#e74c3c', '#c0392b', '#3498db']):
    
    annual = df_clean.groupby('ANNEE')[col].sum()
    ax.bar(annual.index, annual.values, color=color, alpha=0.5, width=0.8)
    rolling = annual.rolling(10, center=True).mean()
    ax.plot(rolling.index, rolling.values, color='black', linewidth=2, label='Moy. glissante 10 ans')
    ax.set_title(title)
    ax.set_xlabel('Année')
    ax.legend()

plt.suptitle("Indicateurs d'extrêmes thermiques par année", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Saisonnalité et profils mensuels

In [ ]:
# Profil saisonnier : température et précipitations par mois
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

mois_labels = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun', 'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']

# Températures par mois (boxplot)
sns.boxplot(data=df_clean, x='MOIS', y='TM', ax=axes[0], palette='RdYlBu_r')
axes[0].set_xticklabels(mois_labels)
axes[0].set_title('Distribution de la température moyenne par mois')
axes[0].set_xlabel('')
axes[0].set_ylabel('°C')

# Précipitations par mois (boxplot)
sns.boxplot(data=df_clean, x='MOIS', y='RR', ax=axes[1], palette='Blues')
axes[1].set_xticklabels(mois_labels)
axes[1].set_title('Distribution des précipitations par mois')
axes[1].set_xlabel('')
axes[1].set_ylabel('mm')

plt.tight_layout()
plt.show()

In [ ]:
# Comparaison des profils saisonniers entre décennies
df_clean['DECENNIE'] = (df_clean['ANNEE'] // 10) * 10
decennies = [1960, 1980, 2000, 2020]
colors_dec = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for dec, color in zip(decennies, colors_dec):
    subset = df_clean[df_clean['DECENNIE'] == dec]
    
    tm_mois = subset.groupby('MOIS')['TM'].mean()
    axes[0].plot(tm_mois.index, tm_mois.values, marker='o', color=color, label=f'{dec}s', linewidth=2)
    
    rr_mois = subset.groupby('MOIS')['RR'].mean()
    axes[1].plot(rr_mois.index, rr_mois.values, marker='o', color=color, label=f'{dec}s', linewidth=2)

axes[0].set_title('Température moyenne par mois — comparaison par décennie')
axes[0].set_ylabel('°C')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(mois_labels)
axes[0].legend()

axes[1].set_title('Précipitations par mois — comparaison par décennie')
axes[1].set_ylabel('mm')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(mois_labels)
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Corrélations entre variables climatiques

In [ ]:
# Matrice de corrélation
corr_cols = ['TM', 'TX', 'TN', 'RR', 'INST', 'FFM', 'NBJGELEE', 'NBJTX30', 'GLOT']
corr_cols = [c for c in corr_cols if c in df_clean.columns]

corr = df_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Matrice de corrélation des variables climatiques', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Analyse géographique — Carte nationale des stations

In [ ]:
# Carte nationale des stations avec température moyenne
stations = df_clean.groupby(['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'DEPARTEMENT']).agg(
    TM_moy=('TM', 'mean'),
    RR_moy=('RR', 'mean'),
    ALTI=('ALTI', 'first'),
    nb_obs=('TM', 'count')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Carte température
scatter1 = axes[0].scatter(stations['LON'], stations['LAT'], c=stations['TM_moy'],
                           cmap='RdYlBu_r', s=15, alpha=0.7, edgecolors='none')
plt.colorbar(scatter1, ax=axes[0], label='Température moyenne (°C)', shrink=0.7)
axes[0].set_title('Température moyenne par station', fontsize=14)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

# Carte précipitations
scatter2 = axes[1].scatter(stations['LON'], stations['LAT'], c=stations['RR_moy'],
                           cmap='Blues', s=15, alpha=0.7, edgecolors='none')
plt.colorbar(scatter2, ax=axes[1], label='Précipitations moyennes (mm/mois)', shrink=0.7)
axes[1].set_title('Précipitations moyennes par station', fontsize=14)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')

plt.suptitle(f'Carte nationale — {len(stations)} stations météo', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Relation altitude / température
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(stations['ALTI'] if 'ALTI' in stations.columns else 
           df_clean.groupby('NUM_POSTE')['ALTI'].first().values,
           stations['TM_moy'], alpha=0.6, color='teal', edgecolors='white', s=50)
ax.set_title("Relation entre altitude et température moyenne", fontsize=14)
ax.set_xlabel("Altitude (m)")
ax.set_ylabel("Température moyenne (°C)")
plt.tight_layout()
plt.show()

In [ ]:
# Température moyenne par département
dept_tm = df_clean.groupby('DEPARTEMENT')['TM'].mean().sort_values()

fig, ax = plt.subplots(figsize=(16, 8))
colors = plt.cm.RdYlBu_r(np.linspace(0, 1, len(dept_tm)))
ax.barh(dept_tm.index.astype(str), dept_tm.values, color=colors)
ax.set_xlabel('Température moyenne (°C)')
ax.set_title('Température moyenne par département', fontsize=14)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Tendance de réchauffement par département (pente linéaire °C/décennie)
from scipy import stats

tendances = {}
for dept, grp in df_clean.groupby('DEPARTEMENT'):
    annual_tm = grp.groupby('ANNEE')['TM'].mean().dropna()
    if len(annual_tm) >= 10:
        slope, _, _, p_value, _ = stats.linregress(annual_tm.index, annual_tm.values)
        tendances[dept] = {'pente_decade': slope * 10, 'p_value': p_value}

tendances_df = pd.DataFrame(tendances).T.sort_values('pente_decade', ascending=True)

fig, ax = plt.subplots(figsize=(16, 8))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in tendances_df['pente_decade']]
ax.barh(tendances_df.index.astype(str), tendances_df['pente_decade'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Tendance (°C / décennie)')
ax.set_title('Tendance de réchauffement par département (régression linéaire)', fontsize=14)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.show()

print(f"Réchauffement moyen national : {tendances_df['pente_decade'].mean():.3f} °C/décennie")

## 10. Anomalies thermiques par rapport à la référence climatologique

In [ ]:
# Anomalies de température par rapport à la période de référence 1961-1990
ref = df_clean[(df_clean['ANNEE'] >= 1961) & (df_clean['ANNEE'] <= 1990)]
tm_ref = ref.groupby('MOIS')['TM'].mean()

df_clean['TM_ANOMALIE'] = df_clean.apply(
    lambda row: row['TM'] - tm_ref.get(row['MOIS'], np.nan) if pd.notna(row['TM']) else np.nan, axis=1)

anomalie_annuelle = df_clean.groupby('ANNEE')['TM_ANOMALIE'].mean()

fig, ax = plt.subplots(figsize=(16, 6))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in anomalie_annuelle.values]
ax.bar(anomalie_annuelle.index, anomalie_annuelle.values, color=colors, alpha=0.7, width=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Anomalies de température moyenne annuelle\n(par rapport à la référence 1961–1990)', fontsize=14)
ax.set_xlabel('Année')
ax.set_ylabel('Anomalie (°C)')
plt.tight_layout()
plt.show()

## 11. Heatmap mois × année

In [ ]:
# Heatmap température moyenne mois × année
pivot_tm = df_clean.pivot_table(values='TM', index='MOIS', columns='ANNEE', aggfunc='mean')

fig, ax = plt.subplots(figsize=(20, 6))
sns.heatmap(pivot_tm, cmap='RdYlBu_r', ax=ax, cbar_kws={'label': '°C'})
ax.set_yticklabels(mois_labels, rotation=0)
ax.set_title('Température moyenne mensuelle par année (toutes stations)', fontsize=14)
ax.set_xlabel('Année')
ax.set_ylabel('')

# Afficher une année sur 5
xticks = ax.get_xticks()
xlabels = [label.get_text() for label in ax.get_xticklabels()]
ax.set_xticks(xticks[::5])
ax.set_xticklabels(xlabels[::5], rotation=45)

plt.tight_layout()
plt.show()

## 12. Synthèse et conclusions de l'EDA

### Observations clés :

1. **Réchauffement climatique visible à l'échelle nationale** : La tendance linéaire montre une hausse significative de la température moyenne sur la période 1950–2026 sur l'ensemble des départements français. Les anomalies positives dominent clairement depuis les années 1990.

2. **Disparités régionales** : Les départements du sud et de basse altitude présentent des températures moyennes plus élevées. La tendance de réchauffement varie selon les départements.

3. **Augmentation des extrêmes thermiques** : Le nombre de jours de forte chaleur (≥30°C et ≥35°C) augmente au fil des décennies, tandis que le nombre de jours de gelée diminue — phénomène visible sur tout le territoire.

4. **Saisonnalité marquée** : Le profil saisonnier classique est bien visible. La comparaison par décennie montre un décalage vers le haut des températures à tous les mois de l'année.

5. **Précipitations** : La tendance est moins nette que pour les températures, mais on observe une variabilité inter-annuelle importante.

6. **Effet de l'altitude** : Corrélation négative nette entre altitude et température, confirmant le gradient thermique altitudinal.

### Prochaines étapes :
- Modélisation prédictive (ARIMA, Prophet, ML) pour la prévision météo
- Visualisations interactives avec Plotly/Dash pour le data storytelling
- Carte interactive par département avec évolution temporelle